In [1]:
# Custom modules
import util.review_util as review_util
import util.model_util as model_util

In [2]:
text_reviews, star_reviews = review_util.get_reviews()
text_reviews = review_util.add_text_features(text_reviews)

In [3]:
# review_util.get_lowest_tfidf_terms(text_reviews.LemmaDocument, 50)

In [4]:
topic_model = model_util.get_bertopic()
review_topics, probs = topic_model.fit_transform(list(text_reviews.text))

# Add the predicted topics to your original dataframe
text_reviews['TopicID'] = review_topics

topic_model.get_topic_info()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,131,-1_pickups_complaints_cans_drive,"[pickups, complaints, cans, drive, road, dumps...","[I would like to make others aware, in case yo..."
1,0,250,0_dumpster_cans_scheduled_call,"[dumpster, cans, scheduled, call, emptied, cus...",[UPDATE* 12/26/2023 This is now the 3rd time t...
2,1,87,1_thankful_grateful_happy_wonderful,"[thankful, grateful, happy, wonderful, appreci...","[Very polite, friendly and accommodating, th..."
3,2,73,2_appreciative_shes_supportive_personable,"[appreciative, shes, supportive, personable, s...",[Danielle was extremely helpful to me! I calle...
4,3,63,3_wave honk_honk_driving_smile wave,"[wave honk, honk, driving, smile wave, park, f...",[Brandon is incredible! He is always so nice a...
5,4,60,4_payments_payment_auto pay_charges,"[payments, payment, auto pay, charges, fees, c...",[These guys have a monopoly and they know it. ...
6,5,44,5_customers_services_staff_paying bill,"[customers, services, staff, paying bill, cont...",[Murrey’s customer service is by far the best....
7,6,40,6_answered quickly_quick response_knowledgeabl...,"[answered quickly, quick response, knowledgeab...",[The ladies that I spoke with were very profes...
8,7,26,7_customers_contact_regards_receptionist,"[customers, contact, regards, receptionist, to...",[5 stars 🌟 Issac L Customer Service Manager!!!...
9,8,22,8_negative reviews_complain_reviews_read reviews,"[negative reviews, complain, reviews, read rev...","[More than once, like most people I'm sure, I ..."


In [5]:
# topic_model.get_topic_info().Representative_Docs[2]

In [6]:
topics_per_class = topic_model.topics_per_class(
    docs=list(text_reviews.text), classes=list(text_reviews.TopicID))

print("Topics (by Class): ")
for i in topics_per_class.index:
    t = topics_per_class.iloc[i]
    print(t.Topic, ": ", t.Words)

Topics (by Class): 
0 :  dumpster, dump, neighbors, call, yard
1 :  thankful, grateful, happy, wonderful, appreciated
2 :  appreciative, shes, supportive, personable, services
3 :  wave honk, honk, driving, smile wave, park
4 :  payments, payment, auto pay, charges, fees
5 :  customers, services, staff, contact, timely
6 :  answered quickly, quick response, knowledgeable, kind, respond
7 :  customers, contact, regards, receptionist, dealing
8 :  negative reviews, complain, reviews, read reviews, yard
9 :  billing, autopay, incredible, agent, payment
-1 :  pickups, complaints, cans, drive, road


In [7]:
hierarchical_topics = topic_model.hierarchical_topics(list(text_reviews.text))
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

100%|██████████| 9/9 [00:02<00:00,  3.62it/s]


.
├─■──answered quickly_quick response_knowledgeable_kind_respond ── Topic: 6
└─services_calling_call_payment_business
     ├─honk_wonderful_appreciative_reviews_taking care
     │    ├─■──thankful_grateful_happy_wonderful_appreciated ── Topic: 1
     │    └─driving_honk_wonderful_appreciative_outstanding
     │         ├─■──wave honk_honk_driving_smile wave_park ── Topic: 3
     │         └─■──negative reviews_complain_reviews_read reviews_yard ── Topic: 8
     └─customers_services_calling_call_payment
          ├─customers_services_calling_call_neighborhood
          │    ├─customers_services_calling_scheduled_call
          │    │    ├─■──customers_services_staff_paying bill_contact ── Topic: 5
          │    │    └─services_scheduled_call_cans_calling
          │    │         ├─■──dumpster_cans_scheduled_call_emptied ── Topic: 0
          │    │         └─■──payments_payment_auto pay_charges_fees ── Topic: 4
          │    └─customers_contact_services_handled_shes
          │      

In [8]:
# Force-assign the noise reviews to the closest topic
text_reviews_df = text_reviews.reset_index(drop=True)
new_topics = topic_model.reduce_outliers(list(text_reviews.text), review_topics, threshold=0.1)

# # topic_model.merge_topics(docs=list(text_reviews.text),
# #                          topics_to_merge=[[0,4], [2,3]])

topic_model.get_topic_info()[['Topic', 'Count']]

,Topic,Count
0,-1,131
1,0,250
2,1,87
3,2,73
4,3,63
5,4,60
6,5,44
7,6,40
8,7,26
9,8,22


In [9]:
topic_model.update_topics(list(text_reviews.text), topics=new_topics,
                          ctfidf_model=model_util.get_ctfidf(), vectorizer_model=model_util.get_vectorizer())
topic_model.get_topic_info()

2026-05-04 23:28:37,072 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,36,-1_pickups_doesnt_missed_dont show,"[pickups, doesnt, missed, dont show, picking, ...","[I would like to make others aware, in case yo..."
1,0,251,0_cans_missed_recycle_pay,"[cans, missed, recycle, pay, bin, street, bins...",[UPDATE* 12/26/2023 This is now the 3rd time t...
2,1,109,1_holidays_people_well_hard,"[holidays, people, well, hard, done, folks, us...","[Very polite, friendly and accommodating, th..."
3,2,80,2_problems_resolved_made_super,"[problems, resolved, made, super, knowledgeabl...",[Danielle was extremely helpful to me! I calle...
4,3,67,3_wave_smile_honk_waves,"[wave, smile, honk, waves, comes, respectful, ...",[Brandon is incredible! He is always so nice a...
5,4,66,4_payment_bill_account_worst,"[payment, bill, account, worst, hold, card, ad...",[These guys have a monopoly and they know it. ...
6,5,45,5_far_community_timely_kids,"[far, community, timely, kids, person, reps, l...",[Murrey’s customer service is by far the best....
7,6,79,6_schedule_hes_safety_bobbie,"[schedule, hes, safety, bobbie, people, absolu...",[The ladies that I spoke with were very profes...
8,7,27,7_request_conversation_take care_contact,"[request, conversation, take care, contact, ex...",[5 stars 🌟 Issac L Customer Service Manager!!!...
9,8,25,8_quality_hes_people_joe,"[quality, hes, people, joe, reviews, normal, h...","[More than once, like most people I'm sure, I ..."


In [10]:
topics_per_class = topic_model.topics_per_class(docs=list(text_reviews.text),
                                                classes=list(text_reviews.TopicID))

print("Topics: ")
for i in topics_per_class.index:
    c_topics = topics_per_class.iloc[i]
    print(c_topics.Class, ": ", c_topics.Words)

# topic_model.visualize_topics_per_class(topics_per_class)

Topics: 
0 :  cans, recycle, missed, pay, bin
1 :  holidays, folks, thankful, community, year
2 :  resolved, problems, made, knowledgeable, super
3 :  wave, honk, smile, waves, comes
4 :  payment, bill, account, worst, card
5 :  far, community, timely, kids, respond
6 :  dependable, professional, quick, efficient, prompt
7 :  conversation, request, take care, contact, exceptional
8 :  quality, hes, people, joe, reviews
9 :  changes, answers, account, autopay, change
-1 :  pickups, doesnt, missed, dont show, picking
-1 :  occasions, multiple, understand, terrible, right
-1 :  people, provide, boxes, well, holidays
-1 :  taken, problems, taken care, talk, positive attitude
-1 :  son, wave, heather, smile, smile wave
-1 :  payment, husband, failed, thursday, stuff
-1 :  future, lol, far, update, setting
-1 :  hes, schedule, safety, bobbie, absolute
-1 :  glad, bring, couple, let, enough
-1 :  fill, werent, july, shut, brought
-1 :  broken, account, didnt, walked, change


In [11]:
# # Calculate mean stars for these new topics
topic_summary = text_reviews.groupby('TopicID')['stars'].mean()
print(topic_summary)

TopicID
-1    3.923664
 0    3.352000
 1    4.965517
 2    4.986301
 3    4.920635
 4    2.816667
 5    4.977273
 6    4.850000
 7    4.961538
 8    5.000000
 9    5.000000
Name: stars, dtype: float64


In [12]:
# topic_model.set_topic_labels(constants.topic_labels)
# topic_model.visualize_barchart(custom_labels=True)

In [13]:
# Visualization Idea(s)
# Are customers more likely to leave a text review when they are angry (1-star) or happy (5-star)?
# You can create a visualization comparing the Star Distribution of Text Reviews vs. Star Distribution of Non-Text Reviews. If 1-star ratings almost always include text, it proves that "Administrative Friction" (your Topic #2) is a high-engagement pain point.